# **TypedDict vs Pydantic: A Beginner's Guide**

## **What is TypedDict?**

TypedDict is a Python feature (from the `typing` module) that lets you specify the expected structure of a dictionary.

```python
from typing import TypedDict

class Movie(TypedDict):
    title: str
    year: int
    rating: float

# This is valid
movie = Movie(title="Inception", year=2010, rating=8.8)

# This is also valid (no errors!)
bad_movie = Movie(title=123, year="not a number", rating="good")
```

### **Key Features of TypedDict:**
- ✅ **Type hints only** - Helps your IDE and code editors
- ✅ **Self-documenting** - Makes code more readable
- ✅ **Lightweight** - No performance overhead
- ❌ **No runtime validation** - Won't catch errors when code runs
- ❌ **No type enforcement** - Can put wrong types and Python won't care

---

## **What is Pydantic?**

Pydantic is a library that provides data validation and settings management using Python type annotations.

```python
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., min_length=1)
    year: int = Field(..., gt=1900, lt=2026)
    rating: float = Field(..., ge=0, le=10)

# This works fine
movie = Movie(title="Inception", year=2010, rating=8.8)

# This raises a validation error!
bad_movie = Movie(title=123, year="not a number", rating="good")
# Error: title must be string, year must be integer, etc.
```

### **Key Features of Pydantic:**
- ✅ **Runtime validation** - Catches errors when code runs
- ✅ **Type enforcement** - Ensures data matches your types
- ✅ **Automatic parsing** - Can convert strings to ints, etc.
- ✅ **Custom validators** - Add your own validation rules
- ✅ **JSON schema generation** - Great for APIs
- ❌ **More overhead** - Slightly slower (usually negligible)
- ❌ **Extra dependency** - Need to install it

---

## **Key Differences**

| Feature | TypedDict | Pydantic |
|---------|-----------|----------|
| **Type** | Built-in Python module | External library |
| **Validation** | ❌ None (hints only) | ✅ Runtime validation |
| **Error handling** | ❌ No errors | ✅ Clear error messages |
| **Type conversion** | ❌ No | ✅ Automatic (str→int, etc.) |
| **Performance** | ⚡ Lightning fast | 🚀 Fast (some overhead) |
| **IDE support** | ✅ Good | ✅ Excellent |
| **Required fields** | ✅ (with `required` or `...`) | ✅ (with `Field(...)`) |
| **Default values** | ✅ Yes | ✅ Yes |
| **Nested models** | ⚠️ Limited | ✅ Full support |
| **Custom validation** | ❌ No | ✅ Yes |
| **JSON schema** | ❌ Manual work | ✅ Automatic |
| **Use case** | Documentation only | Data that MUST be correct |

---

## **When to Use Each**

### **Choose TypedDict when:**

1. **You're documenting internal APIs**
   ```python
   # Just want to show what shape this dict should have
   def process_user(user: UserDict) -> None:
       # Function expects dict with name, age, email
       pass
   ```

2. **Performance is absolutely critical**
   - Millions of operations where microseconds matter
   - You trust your data sources completely

3. **Working with simple scripts or prototypes**
   - Quick and dirty solutions
   - When you don't want extra dependencies

4. **You only need IDE autocomplete**
   - Making code more readable for developers
   - Data comes from trusted sources (internal databases)

### **Choose Pydantic when:**

1. **Data comes from untrusted sources**
   ```python
   # User input from a form - MUST validate!
   user_data = request.get_json()
   user = UserModel(**user_data)  # Validates everything!
   ```

2. **Building APIs or services**
   - Request/response validation
   - Automatic OpenAPI/Swagger docs

3. **Configuration management**
   ```python
   class Settings(BaseModel):
       database_url: str
       api_key: str = Field(..., min_length=32)
       debug_mode: bool = False
   ```

4. **Data needs to be transformed**
   - Convert "123" to integer 123 automatically
   - Parse dates from strings

5. **You need custom validation rules**
   ```python
   @validator('email')
   def validate_email(cls, v):
       if '@' not in v:
           raise ValueError('Invalid email')
       return v
   ```

6. **Working with databases (SQLModel, Beanie)**
   - Many ORMs use Pydantic under the hood

---

## **Quick Decision Guide**

| Your Situation | TypedDict | Pydantic |
|----------------|-----------|----------|
| "I just want autocomplete" | ✅ **Perfect** | ⚠️ Overkill |
| "Users might send bad data" | ❌ Dangerous | ✅ **Must have** |
| "Building a quick script" | ✅ **Good** | ⚠️ Maybe later |
| "Production API" | ❌ Risky | ✅ **Essential** |
| "Processing 1M items/sec" | ✅ **Best choice** | ⚠️ Consider carefully |
| "Forms and user input" | ❌ No validation | ✅ **Perfect** |
| "Learning/teaching" | ✅ Good start | ✅ Great next step |

---

## **Simple Rule of Thumb**

**TypedDict** = "I'm telling my IDE what this dictionary should look like"

**Pydantic** = "I'm telling my PROGRAM what this data MUST look like, and I want it to enforce the rules"

---

## **Code Comparison**

```python
# TYPEDICT - Documentation only
from typing import TypedDict

class User(TypedDict):
    name: str
    age: int

def greet(user: User) -> str:
    return f"Hello {user['name']}"

# This runs fine but crashes later!
user = {"name": 123, "age": "twenty"}  
greet(user)  # Crashes when trying to use 123 as string


# PYDANTIC - Validation and safety
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int

# This raises error immediately
user = User(name=123, age="twenty")  
# ValidationError: name must be string, age must be integer
```

---

## **Conclusion**

- **TypedDict** is like a **shopping list** - tells you what to buy but doesn't check your cart
- **Pydantic** is like a **cashier** - checks every item and refuses bad ones

Start with TypedDict for simple scripts, upgrade to Pydantic when your data needs protection!